In [ ]:
!pip install -q sacrebleu

import re
import math
from collections import Counter
import numpy as np
import pandas as pd
from sacrebleu.metrics import CHRF

In [ ]:
sentences_oare_path = "/kaggle/input/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv"
train_path = "/kaggle/input/deep-past-initiative-machine-translation/train.csv"

sentences_oare = pd.read_csv(sentences_oare_path)
train = pd.read_csv(train_path)

In [ ]:
train['transliteration_length'] = train.apply(lambda row: len(row['transliteration'].split(' ')), axis=1)
train['translation_length'] = train.apply(lambda row: len(row['translation'].split(' ')), axis=1)
train.head()

In [ ]:
test = train.merge(sentences_oare, left_on='oare_id', right_on='text_uuid', suffixes=('_train', '_sentcs'))
print(test.shape, len(test['oare_id'].unique()))
test.to_csv("sentence_match_full.csv", index=False)
test.head()

In [ ]:
df = test.copy()

chrf = CHRF(word_order=2)

def split_translation_clauses(text):
    # conservative clause splitting
    return [t.strip() for t in text.replace(";", ".").split(".") if t.strip()]

aligned_pairs = []

for oare_id, g in df.groupby("oare_id"):
    g = g.sort_values("first_word_number")

    # tokenized transliteration (full tablet)
    full_translit_tokens = g.iloc[0]["transliteration"].split()
    translation_full = g.iloc[0]["translation_train"]
    translation_clauses = split_translation_clauses(translation_full)

    # sentence boundaries
    word_starts = g["first_word_number"].tolist()
    word_starts.append(len(full_translit_tokens) + 1)

    for i, row in g.iterrows():
        start = row["first_word_number"] - 1
        end = word_starts[g.index.get_loc(i) + 1] - 1

        translit_sentence = " ".join(full_translit_tokens[start:end]).strip()
        anchor = str(row["translation_sentcs"])

        if not translit_sentence or not anchor:
            continue

        # fuzzy match translation clauses
        scores = []
        for j, clause in enumerate(translation_clauses):
            score = chrf.sentence_score(anchor, [clause]).score
            scores.append((score, j))

        if not scores:
            continue

        best_idx = max(scores)[1]

        # try expanding window
        candidate = translation_clauses[best_idx]
        if best_idx + 1 < len(translation_clauses):
            expanded = candidate + ", " + translation_clauses[best_idx + 1]
            if chrf.sentence_score(anchor, [expanded]).score > \
               chrf.sentence_score(anchor, [candidate]).score:
                candidate = expanded

        aligned_pairs.append((oare_id, translit_sentence, candidate, anchor))

print(f"Aligned sentence pairs: {len(aligned_pairs)}")
aligned_pairs[:3]

In [ ]:
res = pd.DataFrame(aligned_pairs, columns=["oare_id", "translit_sent", "translation_sent_train", "translation_anchor"])
res.to_csv("sentence_alignment.csv", index=False)